# Uscite ed entrate dall'S&P 500: è solo un effetto del rendimento passato?

**Domanda.** La letteratura (Arnott, Kalesnik & Wu 2018; Cai & Houge 2008) trova che nei 12 mesi dopo un cambio dell'indice le **uscite** rendono più delle **entrate**. Ma le uscite sono loser e piccole, le entrate winner e grandi. La differenza viene dall'indice o solo da queste caratteristiche?

**Metodo.** Ogni evento viene confrontato con azioni "gemelle": **stesso mese, stesso decile di rendimento passato, stesso decile di dimensione**, nessun evento sull'indice in quel mese.

- **Rendimento anomalo** = rendimento dell'evento nei 12 mesi dopo − rendimento medio dei gemelli negli stessi 12 mesi.
- Se è circa 0, l'uscita/entrata non aggiunge niente: conta solo la storia del titolo.
- Il test si fa due volte: con il rendimento passato a **12 mesi** (momentum) e a **5 anni** (reversal a lungo termine).
- Alla fine, la regressione sui fattori Fama-French controlla anche il **value**.

**Convenzioni.** t = mese dell'entrata/uscita effettiva. Detenzione = mesi da t+1 a t+12. Eventi dal 1980 al 2022.

**Dati.** Tutti i file vanno nella cartella `Data/` del progetto. Il notebook la cerca da solo e non modifica nulla.

## 0. Impostazioni

In [ ]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA = next(p / "Data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data").is_dir())

HOLD = 12                      # mesi di detenzione dopo l'evento
FIRST_YEAR, LAST_YEAR = 1980, 2022
N_DEC = 10                     # decili


def find_file(*names):
    """Primo file esistente tra i nomi indicati (nome descrittivo o nome originale WRDS)."""
    for name in names:
        if (DATA / name).is_file():
            return DATA / name
    raise FileNotFoundError(f"Nessuno di questi file è in {DATA}: {names}")


print("Cartella dati:", DATA)

## 1. Rendimenti mensili CRSP: solo azioni ordinarie USA

Filtri sulle azioni (gli stessi del notebook 01): azioni ordinarie (`EQTY`, `COM`, `NS`), società USA, quotate su NYSE, AMEX o Nasdaq.

Nel formato CRSP v2 `MthRet` include già il rendimento di delisting. `MthCap` è in migliaia di dollari.

In [ ]:
crsp = pd.read_csv(find_file("crsp_monthly_stock_returns.csv", "jhvzegjsguw3gw3k.csv"),
                   usecols=["PERMNO", "MthCalDt", "MthRet", "MthCap"], parse_dates=["MthCalDt"])
crsp = crsp.rename(columns={"PERMNO": "permno", "MthCalDt": "date", "MthRet": "ret", "MthCap": "cap"})
crsp = crsp.drop_duplicates(["permno", "date"])

names = pd.read_csv(find_file("crsp_security_history.csv", "honhxqxgerziffrf.csv"),
                    usecols=["permno", "secinfostartdt", "secinfoenddt", "securitytype", "securitysubtype",
                             "sharetype", "usincflg", "primaryexch"],
                    parse_dates=["secinfostartdt", "secinfoenddt"])
ok = names[names["securitytype"].eq("EQTY") & names["securitysubtype"].eq("COM") & names["sharetype"].eq("NS")
           & names["usincflg"].eq("Y") & names["primaryexch"].isin(["N", "A", "Q"])]

# per ogni riga di rendimento, è un'azione ordinaria USA in quella data?
crsp = pd.merge_asof(crsp.sort_values("date"), ok[["permno", "secinfostartdt", "secinfoenddt"]].sort_values("secinfostartdt"),
                     left_on="date", right_on="secinfostartdt", by="permno")
crsp["common"] = crsp["date"].le(crsp["secinfoenddt"])
crsp = crsp.drop(columns=["secinfostartdt", "secinfoenddt"])

crsp["month"] = crsp["date"].dt.to_period("M")
crsp["mnum"] = crsp["date"].dt.year * 12 + crsp["date"].dt.month      # mese come numero, per i controlli
crsp = crsp.sort_values(["permno", "month"], ignore_index=True)
print(f"{len(crsp):,} righe, {crsp['permno'].nunique():,} azioni, dal {crsp['month'].min()} al {crsp['month'].max()}")
print(f"di cui azioni ordinarie USA: {crsp['common'].mean():.0%} delle righe")

## 2. Rendimento passato, dimensione e rendimento futuro

Per ogni azione e mese t:
- `mom` = rendimento cumulato da t−12 a t−2 (momentum; servono tutti gli 11 mesi);
- `ltr` = rendimento cumulato da t−60 a t−13 (lungo termine; servono tutti i 48 mesi);
- `size` = capitalizzazione a fine t−1;
- `fwd` = rendimento cumulato da t+1 a t+12. Se il titolo esce dalla borsa prima, i soldi restano in cassa.

Trucco di calcolo: la somma cumulata dei log-rendimenti (`cs`) permette di ottenere la somma su qualsiasi finestra come differenza di due valori.

In [ ]:
g = crsp.groupby("permno")
log_ret = np.log1p(crsp["ret"])
crsp["cs"] = log_ret.fillna(0).groupby(crsp["permno"]).cumsum()      # somma cumulata dei log-rendimenti
crsp["cn"] = log_ret.notna().groupby(crsp["permno"]).cumsum()        # quanti rendimenti validi finora


def window(far, near):
    """Rendimento cumulato da t-far a t-near; NaN se manca anche un solo mese."""
    total = g["cs"].shift(near) - g["cs"].shift(far + 1)
    count = g["cn"].shift(near) - g["cn"].shift(far + 1)
    contiguous = (crsp["mnum"] - g["mnum"].shift(far + 1)).eq(far + 1)
    return np.expm1(total.where(count.eq(far - near + 1) & contiguous))


crsp["mom"] = window(12, 2)
crsp["ltr"] = window(60, 13)
crsp["size"] = g["cap"].shift(1).where((crsp["mnum"] - g["mnum"].shift(1)).eq(1))

# rendimento futuro: dopo l'ultima riga del titolo (delisting) il valore resta fermo = cassa
last_cs = g["cs"].transform("last")
crsp["fwd"] = np.expm1(g["cs"].shift(-HOLD).fillna(last_cs) - crsp["cs"])
crsp[["mom", "ltr", "size", "fwd"]].describe().T

## 3. Decili ogni mese

Ogni mese, tra le azioni ordinarie USA, ogni azione riceve tre etichette da 1 a 10: decile di `mom`, decile di `ltr`, decile di `size`. 1 = più basso, 10 = più alto.

In [ ]:
def deciles(col):
    pct = crsp.loc[crsp["common"]].groupby("month")[col].rank(pct=True)
    return np.ceil(pct * N_DEC).clip(1, N_DEC)


for col in ["mom", "ltr", "size"]:
    crsp[f"d_{col}"] = deciles(col)

print("Azioni per casella (decile mom x decile size), media mensile:",
      round(crsp.dropna(subset=["d_mom", "d_size"]).groupby(["month", "d_mom", "d_size"]).size().mean(), 1))

## 4. Gli eventi: entrate e uscite dall'S&P 500

- **Entrata** = inizio di un periodo nell'indice.
- **Uscita** = fine di un periodo, se l'azienda non rientra entro 5 giorni.
- Le uscite devono essere ancora quotate a t+1. Altrimenti sono acquisizioni, non uscite per calo di valore.

In [ ]:
spells = pd.read_csv(find_file("sp500_constituent_changes.csv"), parse_dates=["SP500 Start", "SP500 End"])
spells = spells.rename(columns={"PERMNO": "permno", "SP500 Start": "start", "SP500 End": "end"})
spells = spells[["permno", "start", "end"]].drop_duplicates().sort_values(["permno", "start"])
end_of_file = spells["end"].max()

next_start = spells.groupby("permno")["start"].shift(-1)
real_exit = spells["end"].lt(end_of_file) & ~(next_start - spells["end"]).dt.days.le(5)

ev = pd.concat([
    spells[["permno", "start"]].rename(columns={"start": "date"}).assign(type="entrata"),
    spells.loc[real_exit, ["permno", "end"]].rename(columns={"end": "date"}).assign(type="uscita"),
], ignore_index=True)
ev["month"] = ev["date"].dt.to_period("M")
ev = ev[ev["month"].dt.year.between(FIRST_YEAR, LAST_YEAR)]

# attacca le caratteristiche dell'azione nel mese dell'evento
cols = ["permno", "month", "mom", "ltr", "size", "fwd", "d_mom", "d_ltr", "d_size"]
ev = ev.merge(crsp[cols], on=["permno", "month"], how="left")
ev["traded_t1"] = ev.merge(crsp.assign(month=crsp["month"] - 1)[["permno", "month", "ret"]],
                           on=["permno", "month"], how="left")["ret"].notna().to_numpy()
ev = ev[ev["type"].eq("entrata") | ev["traded_t1"]].reset_index(drop=True)
ev["period"] = pd.cut(ev["month"].dt.year, [1979, 1999, 2009, 2017, 2022],
                      labels=["1980–1999", "2000–2009", "2010–2017", "2018–2022"])

print(ev.groupby("type").size())
print("\nDove stanno gli eventi (decile medio, 1 = più basso):")
ev.groupby("type")[["d_mom", "d_ltr", "d_size"]].mean()

## 5. Abbinamento con i gemelli

Per ogni evento: le azioni dello **stesso mese**, **stesso decile di rendimento passato** e **stesso decile di dimensione**, che in quel mese non hanno eventi sull'indice.

La funzione restituisce, per ogni evento, il rendimento medio dei gemelli a 12 mesi e il loro rendimento passato medio. Quest'ultimo serve a controllare che l'abbinamento abbia funzionato: deve essere simile a quello dell'evento.

In [ ]:
event_keys = set(zip(ev["permno"], ev["month"]))
is_event = pd.MultiIndex.from_arrays([crsp["permno"], crsp["month"]]).isin(list(event_keys))
pool = crsp[crsp["common"] & ~is_event]


def match(past):
    """Abbina ogni evento ai gemelli con lo stesso decile di `past` ('mom' o 'ltr') e di size."""
    keys = ["month", f"d_{past}", "d_size"]
    twins = (pool.dropna(subset=[f"d_{past}", "d_size", "fwd"])
                 .groupby(keys).agg(twin_fwd=("fwd", "mean"), twin_past=(past, "mean"), n_twins=("fwd", "size"))
                 .reset_index())
    out = ev.dropna(subset=[f"d_{past}", "d_size", "fwd"]).merge(twins, on=keys, how="inner")
    out["abn"] = out["fwd"] - out["twin_fwd"]          # rendimento anomalo a 12 mesi
    return out


def summary(m, past):
    def stats(x):
        a = x["abn"]
        return pd.Series({
            "N": len(x),
            "gemelli per evento": x["n_twins"].mean(),
            f"{past} passato evento": x[past].mean(),
            f"{past} passato gemelli": x["twin_past"].mean(),
            "rend. 12m evento": x["fwd"].mean(),
            "rend. 12m gemelli": x["twin_fwd"].mean(),
            "anomalo medio": a.mean(),
            "anomalo mediano": a.median(),
            "% anomalo > 0": 100 * (a > 0).mean(),
            "t-stat": a.mean() / a.std() * np.sqrt(len(a)),
        })
    table = m.groupby("type").apply(stats, include_groups=False).T
    # uscite - entrate: differenza tra le medie dei rendimenti anomali
    u, e = m.loc[m["type"].eq("uscita"), "abn"], m.loc[m["type"].eq("entrata"), "abn"]
    diff = u.mean() - e.mean()
    se = np.sqrt(u.var() / len(u) + e.var() / len(e))
    print(f"Uscite − entrate, rendimento anomalo: {diff:.3f}  (t = {diff / se:.2f})")
    return table

## 6. Risultati: abbinamento sul rendimento a 12 mesi (momentum)

Le t-stat a livello di evento sono ottimistiche, perché eventi dello stesso periodo sono correlati. Il test più rigoroso è quello del punto 8.

In [ ]:
m_mom = match("mom")
display(summary(m_mom, "mom"))
display(m_mom.groupby(["type", "period"], observed=True)["abn"].agg(["count", "mean", "median"]))

## 7. Risultati: abbinamento sul rendimento a 5 anni (reversal a lungo termine)

In [ ]:
m_ltr = match("ltr")
display(summary(m_ltr, "ltr"))
display(m_ltr.groupby(["type", "period"], observed=True)["abn"].agg(["count", "mean", "median"]))

## 8. Portafogli e fattori Fama-French (controlla anche il value)

Ogni mese il portafoglio contiene tutti gli eventi dei 12 mesi precedenti. Per ogni evento si prende il suo rendimento di quel mese **meno** il rendimento medio dei suoi gemelli nello stesso mese. Poi si fa la media sugli eventi.

Si regredisce questa serie mensile sui fattori Fama-French 5 + momentum, con errori standard di Newey-West. Se l'**alpha** resta positivo e significativo, la differenza non si spiega né con il rendimento passato, né con la dimensione, né con il value.

Servono in `Data/` i file `F-F_Research_Data_5_Factors_2x3_CSV.zip` e `F-F_Momentum_Factor_CSV.zip`, da lasciare compressi.

In [ ]:
def read_french(zip_name, columns):
    """Legge uno zip di Kenneth French e tiene le righe mensili (AAAAMM)."""
    with zipfile.ZipFile(DATA / zip_name) as z:
        lines = z.read(z.namelist()[0]).decode("latin1").splitlines()
    header = next(i for i, l in enumerate(lines) if set(columns) <= {c.strip() for c in l.split(",")})
    rows = []
    for l in lines[header + 1:]:
        parts = [x.strip() for x in l.split(",")]
        if not (len(parts[0]) == 6 and parts[0].isdigit()):
            break
        rows.append(parts)
    names = ["month"] + [c.strip() for c in lines[header].split(",")[1:]]
    df = pd.DataFrame(rows, columns=names[:len(rows[0])])
    df["month"] = pd.PeriodIndex(df["month"], freq="M")
    return df.set_index("month")[columns].astype(float) / 100


factors = read_french("F-F_Research_Data_5_Factors_2x3_CSV.zip", ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]).join(
    read_french("F-F_Momentum_Factor_CSV.zip", ["Mom"]), how="inner")

# rendimento mensile medio dei gemelli per casella (mese di formazione, decili) e mese di detenzione
ret_long = crsp[["permno", "month", "ret"]].dropna()


def monthly_abnormal(m, past):
    keys = ["month", f"d_{past}", "d_size"]
    cells = pool.dropna(subset=[f"d_{past}", "d_size"])[keys + ["permno"]]
    rows = []
    for k in range(1, HOLD + 1):
        # rendimento di eventi e gemelli k mesi dopo la formazione
        r = ret_long.assign(month=ret_long["month"] - k)
        twin_k = cells.merge(r, on=["permno", "month"]).groupby(keys)["ret"].mean().rename("twin_ret")
        ev_k = m[["permno", "type"] + keys].merge(r, on=["permno", "month"])
        ev_k = ev_k.join(twin_k, on=keys).assign(hold_month=lambda d: d["month"] + k)
        rows.append(ev_k)
    x = pd.concat(rows)
    x["abn"] = x["ret"] - x["twin_ret"]
    series = x.groupby(["hold_month", "type"])["abn"].mean().unstack()
    series["uscite − entrate"] = series["uscita"] - series["entrata"]
    return series


def alpha_table(series):
    out = {}
    for col in series.columns:
        d = pd.concat([series[col].rename("y"), factors], axis=1, join="inner").dropna()
        fac = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]
        fit = sm.OLS(d["y"], sm.add_constant(d[fac])).fit(cov_type="HAC", cov_kwds={"maxlags": HOLD})
        out[col] = {"mesi": len(d), "media %/mese": 100 * d["y"].mean(),
                    "alpha FF5+Mom %/mese": 100 * fit.params["const"], "t alpha": fit.tvalues["const"],
                    "beta SMB": fit.params["SMB"], "beta HML": fit.params["HML"], "beta Mom": fit.params["Mom"]}
    return pd.DataFrame(out).T


for past, m in [("mom", m_mom), ("ltr", m_ltr)]:
    print(f"\nAbbinamento su {past}: rendimento evento − gemelli, per mese")
    display(alpha_table(monthly_abnormal(m, past)))